In [1]:
from woa import WOA

Temporary definition of global variables - they will be extracted based on the datatset.

In [2]:
w1 = w2 = w3 = 1
buy_price = 1
sell_price = 1
target_energy = 1
e_bat = 1
eff_charge = 1
eff_discharge = 1
k_slope = 1
num_household = 30
E_loss = 0.1
E_load = E_pv = E_bat = currSoC = prevSoC = [1 for i in range(num_household)]
batt_cost = batt_capacity = [2 for i in range(num_household)]
maxSoC = [95 for i in range(num_household)]
minSoC = [10 for i in range(num_household)]
l1 = l2 = l3 = l4 = 1
grid_energy_min = -100
grid_energy_max = 300

Compute the energy exchange between a household and the grid.

In [3]:
def getGridForHousehold(houseIdx):
    E_i_grid = 0
    E_i_grid = E_load[houseIdx] - E_pv[houseIdx] + E_bat[houseIdx]
    return E_i_grid

Compute the total exchange with the grid, for the entire VPP.

In [4]:
def getTotalGridEnergy():
    E_grid = 0
    for i in range(num_household):
        E_grid += getGridForHousehold(i)
    E_grid += E_loss
    return E_grid

Get the state of charge for the next time interval.

In [5]:
def computeSoCForBat(idx, e_bat):
    currSoC[idx] = 0
    if (e_bat >= 0):
        currSoC[idx] = prevSoC[idx] + (eff_charge * e_bat)
    else:
        currSoC[idx] = prevSoC[idx] - (eff_discharge * abs(e_bat))

def computeTotalSoC(num_household, e_bat_lst):
    for i in range(num_household):
        computeSoCForBat(i, e_bat_lst[i])

Compute the cost for the current time interval.

In [6]:
def computeCost(grid_energy):
    cost = 0
    if (grid_energy >= 0):
        cost = grid_energy * buy_price
    else:
        cost = abs(grid_energy) * (sell_price) * (-1)
    return cost 

Compute the tracking component for the current time interval.

In [7]:
def computeTrackComponent(grid_energy):
    track = pow((grid_energy - target_energy), 2)  
    return track

Compute the battery penalty function, based on the current and previous SoC.

In [8]:
def computeBattFctForBatt(idx):
    batt_i = abs(k_slope / 100) * ((prevSoC[idx] - currSoC[idx]) / batt_capacity[idx]) * batt_cost[idx]
    return batt_i

def computeTotalBattFct(num_household):
    F_bat = 0
    for i in range(num_household):
        F_bat += computeBattFctForBatt(i)
    return F_bat


The function computeBattPenaltyForBatt computes the penalty for a battery. Apply it to all the batteries to compute the total penalty applied to the batteries.

In [9]:
def computeBattPenaltyForBatt(currSoC, maxSoC, minSoC):
    F_pen_bat_i = l1* pow(max(0, (currSoC - maxSoC)), 2) + l2 *pow(max(0, (minSoC - currSoC)), 2) 
    return F_pen_bat_i

def computeTotalBattPenalty(num_household):
    F_pen_bat = 0
    for i in range(num_household):
        F_pen_bat += computeBattPenaltyForBatt(currSoC[i], maxSoC[i], minSoC[i])
    return F_pen_bat

Compute the grid penalty component.

In [10]:
def computeGridPenalty(E_grid, E_grid_max, E_grid_min):
    F_pen_grid = l3 * pow(max(0, (E_grid - E_grid_max)), 2) + l4 * pow(min(0, (E_grid - E_grid_min)), 2)
    return F_pen_grid

Using the 2 previous components, compute the penalty function, for the current timestep, for the VPP.

In [11]:
def computePenalty(num_household, E_grid, E_grid_max, E_grid_min):
    F_pen_bat = computeTotalBattPenalty(num_household)
    F_pen_grid = computeGridPenalty(E_grid, E_grid_max, E_grid_min)
    F_pen = F_pen_bat + F_pen_grid
    return F_pen

Get the fitness function for an individual

In [12]:
def fitness():
    grid_energy = getTotalGridEnergy()
    f_cost = computeCost(grid_energy)
    f_track = computeTrackComponent(grid_energy)
    f_bat = computeTotalBattFct(num_household)
    f_pen = computePenalty(num_household, grid_energy, grid_energy_max, grid_energy_min)
    fitness = w1 * f_cost + w2 * f_track + w3 * f_bat + f_pen
    return fitness

In [13]:
woa = WOA(100, 500, -500, fitness)


In [14]:
best = woa.computeBest(100, num_household, 500, -500, 100)
print(best.fitness_value)
print(best.transf_energy)

3306.91
[344.42185152504817, 257.95440294030243, -79.428419169155, -241.08324970703666, 11.274721368608539, -95.06586254958569, 283.7985890347726, -196.68727392107257, -23.403045847644194, 83.38203945503119, 408.11288519533514, 4.686855817390267, -218.16215560029616, 255.80420415722392, 118.36899667533157, -249.49365863755946, 409.7462559682401, 482.7854760376531, 310.2172359965896, 402.16595043958273, -189.8524306806674, 229.83174826012862, 398.83828796799344, 183.9839319154413, -27.857284547286667, -399.2987919316342, -65.82816454621633, 110.8869734438016, 413.0110532378982, 466.60636777075877]


,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
Bus,,,,,,,,,,,,,
zone_1,1.0,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,
zone_2,1.0,,0.0,0.0,AC,,,1.0,0.0,inf,PQ,,


Index(['load_2'], dtype='object')